# 새 오류 위치 탐지기 학습 — Gate A
기존 best.pt 획 추출기는 동결하고 새 탐지기를 학습합니다. 아직 자동 수정/확산 연결 단계가 아닙니다.
같은 폰트의 정상 글자에 부착 돌출/분리 잡획을 합성해 정확한 변경 위치를 정답으로 사용합니다.
정상 이미지도 오류 없음으로 학습합니다. 깨끗한 RGB는 새 모델의 입력이 아닙니다.
문자와 입력 폰트를 train/validation/test에서 분리합니다. 기존 추출기의 사전학습 중복은 배제하지 못합니다.
실제 확산 출력과 사진의 성능은 별도 검수해야 합니다.

GPU 런타임에서 1 준비 → 2 기존 체크포인트·자료 → 3 데이터/주석 확인 → 4 학습 → 5 held-out 결과·ZIP → 6 실제 단일 글자 crop 진단(선택).
기본 96문자, 10에폭, batch4는 실험 시작값이며 학습 시간/최적 학습량은 미검증입니다.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile, json
root = Path(tempfile.mkdtemp(prefix='hsqm_diffusion_score_', dir='/content'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
    'https://github.com/yechan25/hsqm.git', str(root)], check=True)
commit = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.35.1', 'transformers==4.49.0', 'accelerate', 'safetensors',
    'timm', 'pandas', 'numpy', 'tqdm', 'matplotlib', 'pillow', 'scipy',
    'scikit-image', 'pyyaml', 'fonttools', 'ipywidgets'], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key.split('.')[0] in ('stroke_model', 'font_test', 'diffusion_model'):
        del sys.modules[key]
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab에서 GPU 런타임으로 바꿔주세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('실행 GitHub 커밋:', commit)
from google.colab import drive, output
drive.mount('/content/drive')
output.enable_custom_widget_manager()
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','fonts-nanum'],check=True)

In [ ]:
from stroke_model.convert_reference_csv import existing_path
from diffusion_model.score_pilot import training_catalog, generate_pilot, score_pilot, load_catalog_reference
from diffusion_model.pilot_data import prepare_pilot_data
from diffusion_model.multiglyph_pilot import generate_multiglyph_pilot
from IPython.display import display
import ipywidgets as widgets
HSQM = existing_path('/content/drive/Shareddrives/2026 자율연구/HSQM')
CHECKPOINT = None  # 폰트 실험에서 사용한 best.pt의 전체 경로를 넣으면 그 모델 사용
DATA_ROOT = HSQM/'dataset/automated_train_dataset'  # 기존 학습 ZIP/TAR가 있는 폴더
CSV_PATH = None  # None: 체크포인트 옆에 저장된 학습 CSV를 자동 사용
TRAIN_MANIFEST = None  # 기존 세션의 유효한 목록이 있을 때만 선택적으로 지정. 없어도 자동 재생성
REFERENCE_ONLY = True  # 시험에는 기준 글자/기준 획만 사용. 전체 학습 NPZ 변환 생략
SCORE_CONFIG = {}  # 폰트 시험에서 값을 바꿨다면 동일한 설정 입력
SOURCE_CONFIG = {}
GLYPH_COUNTS = [4]
CANVAS_SIZE = 576
FIXED_CELL_PIXELS = None  # None: 전체 크기 고정. 192: 글자당 크기 고정 비교
STRENGTHS = [.65]
SEEDS = [100]
STEPS = 100
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
if CHECKPOINT is None:
    candidates = sorted((HSQM/'v2_topology_runs').glob('*/best.pt'))
    if not candidates:
        raise FileNotFoundError('CHECKPOINT에 기존 Topo best.pt를 지정하세요.')
    CHECKPOINT = candidates[-1]
CHECKPOINT = existing_path(CHECKPOINT)
print('사용할 획 모델:', CHECKPOINT, flush=True)
preparation = prepare_pilot_data(CHECKPOINT, DATA_ROOT, csv_path=CSV_PATH, manifest=TRAIN_MANIFEST,
    reference_only=REFERENCE_ONLY)
TRAIN_MANIFEST = Path(preparation['manifest'])
print('데이터 준비 완료. 3번 셀에서 합성 오류 주석을 확인하세요.', flush=True)


EPOCHS=10
BATCH_SIZE=4
WIDTH=32
LEARNING_RATE=2e-4
MAX_CHARACTERS=96
VARIANTS=6
TRAIN_SEED=42
# CHECKPOINT is the EXISTING stroke extractor, not the new error detector.
print('오류 탐지기만 새로 학습하며 기존 획 모델은 동결합니다.')

In [ ]:
from stroke_model.error_synthesis import build_error_sets,glyph_from_entry
from stroke_model.infer_topology import load_topology
catalog=training_catalog(TRAIN_MANIFEST)
font_root=Path('/usr/share/fonts/truetype/nanum')
def find_font(*names):return next((str(font_root/n) for n in names if (font_root/n).is_file()),None)
fonts=dict(train=[find_font('NanumGothic.ttf'),find_font('NanumGothicBold.ttf')],
           val=[find_font('NanumMyeongjo.ttf')],test=[find_font('NanumSquareR.ttf','NanumBarunGothic.ttf')])
fonts={k:[p for p in v if p] for k,v in fonts.items()}
sets,split_metadata=build_error_sets(catalog,fonts,MAX_CHARACTERS,VARIANTS,TRAIN_SEED)
extractor,size,extractor_epoch=load_topology(CHECKPOINT,torch.device('cuda'))
from datetime import datetime
RUN_DIR=HSQM/'error_localizer_runs'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
for split,data in sets.items():print(split,'characters',len(split_metadata['characters'][split]),'samples',len(data),'fonts',fonts[split])
print('문자 정보 확인:',[(glyph_from_entry(e),e.get('reference',e.get('path'))) for e in sets['train'].entries[:8]])
import matplotlib.pyplot as plt
fig,axes=plt.subplots(3,3,figsize=(9,9))
for i in range(3):
    s=sets['train'][i]
    for ax,image,title in zip(axes[i],[s['clean'].permute(1,2,0),s['rgb'].permute(1,2,0),s['extra'][0]],['Clean (label only)','Synthetic input','Extra ink ground truth']):
        ax.imshow(image,cmap='magma',vmin=0,vmax=1);ax.set_title(title);ax.axis('off')
plt.tight_layout();plt.show()
print('4번 실행 전, 원래 획이 extra 정답으로 잡히지 않는지 확인하세요.')

In [ ]:
from stroke_model.train_error_localizer import fit_error_localizer
import hashlib
metadata=dict(split_metadata,extractor_checkpoint=str(CHECKPOINT),
    extractor_sha256=hashlib.sha256(Path(CHECKPOINT).read_bytes()).hexdigest(),git_commit=commit,
    data_manifest=str(TRAIN_MANIFEST))
model,history=fit_error_localizer(extractor,sets['train'],sets['val'],RUN_DIR,device='cuda',
    epochs=EPOCHS,batch_size=BATCH_SIZE,learning_rate=LEARNING_RATE,width=WIDTH,seed=TRAIN_SEED,metadata=metadata)
(RUN_DIR/'git_commit.txt').write_text(commit)
print('새 오류 탐지기:',RUN_DIR/'best.pt')
print('threshold 미보정: 아직 자동 삭제에 연결하지 마세요.')

In [ ]:
from stroke_model.train_error_localizer import evaluate_errors
metrics=evaluate_errors(model,sets['test'],'cuda',BATCH_SIZE)
(RUN_DIR/'test_metrics.json').write_text(json.dumps(metrics,indent=2))
print('held-out test:',metrics)
fig,axes=plt.subplots(3,4,figsize=(12,9))
for i in range(3):
    s=sets['test'][i]
    with torch.no_grad():
        probability=model(s['rgb'][None].cuda(),s['reference'][None].cuda(),s['strokes'][None].cuda(),s['codepoint'][None].cuda())['extra_probability'][0,0].cpu()
    for ax,image,title in zip(axes[i],[s['clean'].permute(1,2,0),s['rgb'].permute(1,2,0),s['extra'][0],probability],['Clean (evaluation only)','Input','Ground truth','Predicted extra']):
        ax.imshow(image,cmap='magma',vmin=0,vmax=1);ax.set_title(title);ax.axis('off')
plt.tight_layout();fig.savefig(RUN_DIR/'heldout_examples.png',dpi=140);plt.show()
fig,ax=plt.subplots()
ax.plot([r['epoch'] for r in history],[r['train_loss'] for r in history],label='train')
ax.plot([r['epoch'] for r in history],[r['validation']['loss'] for r in history],label='validation')
ax.legend();fig.savefig(RUN_DIR/'loss.png');plt.show()
import zipfile
from google.colab import files
archive=RUN_DIR.parent/(RUN_DIR.name+'_error_training.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.iterdir():
        if path.is_file():z.write(path,path.name)
files.download(str(archive))

In [ ]:
# Optional: ONE generated character crop, not the entire 2x2 sheet.
REAL_IMAGE=None
REAL_CHAR='가'
if REAL_IMAGE is not None:
    import numpy as np
    from PIL import Image
    from diffusion_model.score_pilot import load_catalog_reference
    entry=next((e for e in catalog if glyph_from_entry(e)==REAL_CHAR),None)
    if entry is None:raise ValueError('해당 문자의 표준 획 조건이 없습니다.')
    ref,strokes=load_catalog_reference(entry)
    with Image.open(REAL_IMAGE) as im:rgb=np.asarray(im.convert('RGB').resize((size,size)),np.float32)/255
    with torch.no_grad():
        out=model(torch.from_numpy(rgb).permute(2,0,1)[None].cuda(),torch.from_numpy(ref)[None].cuda(),torch.from_numpy(strokes)[None].cuda(),torch.tensor([ord(REAL_CHAR)],device='cuda'))
    fig,axes=plt.subplots(1,2,figsize=(8,4))
    axes[0].imshow(rgb);axes[0].set_title('Generated crop')
    axes[1].imshow(out['extra_probability'][0,0].cpu(),cmap='magma',vmin=0,vmax=1);axes[1].set_title('Extra ink probability (uncalibrated)')
    for ax in axes:ax.axis('off')
    plt.show()